<a href="https://colab.research.google.com/github/ashokmulchandani/CUDA-GPU-Colab-Computer-Vision-Project-Ashok-1/blob/main/6_Multistream.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi | head -5


Fri May 29 21:48:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


In [2]:
%%writefile multistream.cu
#include <stdio.h>
#include <stdlib.h>

#define N 10000000  // 10 million elements per batch
#define NUM_BATCHES 4
#define NUM_STREAMS 4

// Simple kernel: double every value (simulates "processing")
__global__ void process_kernel(float *data, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        // Simulate some work (multiple operations)
        float val = data[i];
        val = val * 2.0f + 1.0f;
        val = sqrtf(val * val + 1.0f);
        data[i] = val;
    }
}

int main() {
    int size = N * sizeof(float);

    printf("=== CUDA Multi-Stream Demo ===\n\n");
    printf("Scenario: Process %d batches of %d elements each\n", NUM_BATCHES, N);
    printf("Each batch: Upload → Process → Download\n\n");

    // Host memory (PINNED — required for async transfers!)
    float *h_data[NUM_BATCHES];
    for (int i = 0; i < NUM_BATCHES; i++) {
        cudaMallocHost(&h_data[i], size);  // pinned memory for async
        for (int j = 0; j < N; j++) h_data[i][j] = (float)(i + 1);
    }

    // Device memory (one buffer per batch)
    float *d_data[NUM_BATCHES];
    for (int i = 0; i < NUM_BATCHES; i++)
        cudaMalloc(&d_data[i], size);

    int threads = 256;
    int blocks = (N + threads - 1) / threads;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float ms;

    // ============================================
    // METHOD 1: Single stream (sequential — SLOW)
    // ============================================
    cudaEventRecord(start);

    for (int i = 0; i < NUM_BATCHES; i++) {
        // Upload → Process → Download (one at a time, waiting between each)
        cudaMemcpy(d_data[i], h_data[i], size, cudaMemcpyHostToDevice);
        process_kernel<<<blocks, threads>>>(d_data[i], N);
        cudaMemcpy(h_data[i], d_data[i], size, cudaMemcpyDeviceToHost);
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    float single_time = ms;

    printf("METHOD 1: Single Stream (sequential)\n");
    printf("  ┌─────────┬──────────┬──────────┐\n");
    printf("  │Upload 1 │Process 1 │Download 1│ → then Upload 2 → ...\n");
    printf("  └─────────┴──────────┴──────────┘\n");
    printf("  Time: %.2f ms\n\n", single_time);

    // Reset data
    for (int i = 0; i < NUM_BATCHES; i++)
        for (int j = 0; j < N; j++) h_data[i][j] = (float)(i + 1);

    // ============================================
    // METHOD 2: Multi-stream (overlapped — FAST!)
    // ============================================

    // Create streams
    cudaStream_t streams[NUM_STREAMS];
    for (int i = 0; i < NUM_STREAMS; i++)
        cudaStreamCreate(&streams[i]);

    cudaEventRecord(start);

    for (int i = 0; i < NUM_BATCHES; i++) {
        int s = i % NUM_STREAMS;  // round-robin across streams

        // All three operations go into the SAME stream
        // But different batches go into DIFFERENT streams → overlap!
        cudaMemcpyAsync(d_data[i], h_data[i], size, cudaMemcpyHostToDevice, streams[s]);
        process_kernel<<<blocks, threads, 0, streams[s]>>>(d_data[i], N);
        cudaMemcpyAsync(h_data[i], d_data[i], size, cudaMemcpyDeviceToHost, streams[s]);
    }

    // Wait for ALL streams to finish
    for (int i = 0; i < NUM_STREAMS; i++)
        cudaStreamSynchronize(streams[i]);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    float multi_time = ms;

    printf("METHOD 2: Multi-Stream (overlapped)\n");
    printf("  Stream 0: ┌─Upload 1─┬─Process 1─┬─Download 1─┐\n");
    printf("  Stream 1: │  ┌─Upload 2─┬─Process 2─┬─Download 2─┐\n");
    printf("  Stream 2: │  │  ┌─Upload 3─┬─Process 3─┬─Download 3─┐\n");
    printf("  Stream 3: │  │  │  ┌─Upload 4─┬─Process 4─┬─Download 4─┐\n");
    printf("  Time: %.2f ms\n\n", multi_time);

    // Results
    printf("═══════════════════════════════════════════\n");
    printf("  Single stream: %.2f ms\n", single_time);
    printf("  Multi-stream:  %.2f ms\n", multi_time);
    printf("  Speedup:       %.2fx faster!\n", single_time / multi_time);
    printf("═══════════════════════════════════════════\n\n");

    printf("WHY it's faster:\n");
    printf("  Single: GPU waits during upload/download (idle time)\n");
    printf("  Multi:  While uploading batch 2, GPU processes batch 1\n");
    printf("          → GPU is NEVER idle!\n\n");

    printf("WHEN to use multi-stream:\n");
    printf("  • Video processing (next frame uploads while current processes)\n");
    printf("  • Batch inference (multiple images in parallel)\n");
    printf("  • Any pipeline with upload → compute → download pattern\n\n");

    printf("KEY REQUIREMENTS:\n");
    printf("  • Pinned memory (cudaMallocHost) — required for async transfers\n");
    printf("  • cudaMemcpyAsync (not cudaMemcpy) — non-blocking copy\n");
    printf("  • kernel<<<blocks, threads, 0, stream>>> — assign kernel to stream\n");
    printf("  • Independent batches (batch 2 doesn't depend on batch 1's result)\n");

    // Cleanup
    for (int i = 0; i < NUM_BATCHES; i++) {
        cudaFreeHost(h_data[i]);
        cudaFree(d_data[i]);
    }
    for (int i = 0; i < NUM_STREAMS; i++)
        cudaStreamDestroy(streams[i]);

    return 0;
}


Writing multistream.cu


In [3]:
!nvcc multistream.cu -o multistream -lm && ./multistream


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
=== CUDA Multi-Stream Demo ===

Scenario: Process 4 batches of 10000000 elements each
Each batch: Upload → Process → Download

METHOD 1: Single Stream (sequential)
  ┌─────────┬──────────┬──────────┐
  │Upload 1 │Process 1 │Download 1│ → then Upload 2 → ...
  └─────────┴──────────┴──────────┘
  Time: 131.62 ms

METHOD 2: Multi-Stream (overlapped)
  Stream 0: ┌─Upload 1─┬─Process 1─┬─Download 1─┐
  Stream 1: │  ┌─Upload 2─┬─Process 2─┬─Download 2─┐
  Stream 2: │  │  ┌─Upload 3─┬─Process 3─┬─Download 3─┐
  Stream 3: │  │  │  ┌─Upload 4─┬─Process 4─┬─Download 4─┐
  Time: 18.60 ms

═══════════════════════════════════════════
  Single stream: 131.62 ms
  Multi-stream:  18.60 ms
  Speedup:       7.07x faster!
═══════════════════════════════════════════

WHY it's faster:
  Single: GPU waits during upload/down